# 013 Model Workbench Start

This notebook opens the runway for downloaded-model work. It audits local model readiness and runs only bounded BGE embedding-classifier smoke samples by default. It does not score holdout, train, call an LLM, or generate answers.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROJECT_PYTHON = ROOT / '.it_support' / 'Scripts' / 'python.exe'
if not PROJECT_PYTHON.exists():
    PROJECT_PYTHON = Path(sys.executable)
READINESS = ROOT / 'data' / 'eval' / 'model_workbench_readiness'
EMBED = ROOT / 'data' / 'eval' / 'embedding_classifier_dev' / 'bge_small_en_v15'

assert (ROOT / 'data' / 'eval' / 'classification_ladder_dev' / 'classification_ladder_summary_dev.json').exists(), 'Run notebook 011 first.'

## Readiness Audit

In [2]:
RUN_READINESS_AUDIT = True

if RUN_READINESS_AUDIT:
    subprocess.run(
        [
            str(PROJECT_PYTHON),
            str(ROOT / 'scripts' / 'audit_model_workbench_readiness.py'),
        ],
        cwd=ROOT,
        check=True,
    )

readiness_summary = READINESS / 'model_workbench_readiness_summary.md'
assert readiness_summary.exists(), f'Missing readiness summary: {readiness_summary}'
display(Markdown(readiness_summary.read_text(encoding='utf-8')))

# Model Workbench Readiness

This audit checks downloaded model assets and runtime packages. It does not load large models, train, score holdout, call an LLM, or generate answers.

## Counts

| models_audited | models_present | ready_sentence_transformer_models | local_llm_assets_present |
| --- | --- | --- | --- |
| 6 | 6 | 2 | 4 |

## Torch Runtime

| package | available | version | cuda_available | cuda_version | device_count | device_name |
| --- | --- | --- | --- | --- | --- | --- |
| torch | True | 2.6.0+cu124 | True | 12.4 | 1 | NVIDIA GeForce RTX 4060 Laptop GPU |

## Model Inventory

| key | role | backend | registry_status | exists | path_type | size_gb |
| --- | --- | --- | --- | --- | --- | --- |
| bge_small_en_v15 | small_cpu_embedding_baseline | sentence-transformers | target | True | dir | 0.125 |
| qwen3_embedding_06b | strong_embedding_challenger | sentence-transformers | target | True | dir | 1.125 |
| gemma4_e2b_it | primary_triage_classifier_base | transformers | target | True | dir | 9.573 |
| qwen35_4b | classifier_challenger | transformers | target | True | dir | 8.701 |
| ministral3_3b_instruct_q4km | tiny_fast_fallback_demo_model | llama.cpp | target | True | dir | 2.783 |
| gemma4_e4b_it_q4km | primary_specialist_generator | llama.cpp | present | True | file | 4.635 |

## Starter Commands

```powershell
.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key bge_small_en_v15 --summary
```
```powershell
.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key bge_small_en_v15 --include-tags --summary
```
```powershell
.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key qwen3_embedding_06b --device auto --require-cuda --batch-size 4 --summary
```
```powershell
.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key bge_small_en_v15 --full-dev --summary
```

## Outputs

- `data\eval\model_workbench_readiness\model_workbench_inventory.csv`
- `data\eval\model_workbench_readiness\model_workbench_runtime.csv`
- `data\eval\model_workbench_readiness\model_workbench_readiness_summary.json`
- `data\eval\model_workbench_readiness\model_workbench_readiness_summary.md`


## BGE Classifier Smoke Samples

In [3]:
RUN_BGE_SMOKE_SAMPLES = True

if RUN_BGE_SMOKE_SAMPLES:
    base_cmd = [
        str(PROJECT_PYTHON),
        str(ROOT / 'scripts' / 'evaluate_embedding_classifier_baseline.py'),
        '--model-key',
        'bge_small_en_v15',
        '--max-eval-cases',
        '32',
        '--max-reference-cases',
        '128',
        '--batch-size',
        '16',
    ]
    subprocess.run(base_cmd, cwd=ROOT, check=True)
    subprocess.run(base_cmd + ['--include-tags'], cwd=ROOT, check=True)

text_summary = EMBED / 'embedding_classifier_summary_dev_sample_text_bge_small_en_v15.json'
metadata_summary = EMBED / 'embedding_classifier_summary_dev_sample_metadata_bge_small_en_v15.json'
assert text_summary.exists(), f'Missing text BGE sample summary: {text_summary}'
assert metadata_summary.exists(), f'Missing metadata BGE sample summary: {metadata_summary}'

## Smoke Results

In [4]:
def sample_row(path):
    summary = json.loads(path.read_text(encoding='utf-8'))
    return {
        'profile': summary['primary_metrics']['profile'],
        'scope': summary['scope'],
        'eval_cases': summary['counts']['eval_cases'],
        'reference_cases': summary['counts']['reference_cases'],
        'device': summary['model']['resolved_device'],
        'primary_accuracy': summary['primary_metrics']['accuracy'],
        'multilabel_micro_f1': summary['multilabel_metrics']['micro_f1'],
        'multilabel_exact_match': summary['multilabel_metrics']['exact_match_ratio'],
        'ranked_primary_hit_at_3': summary['ranked_metrics']['primary_hit_at_3'],
        'graded_ndcg_at_3': summary['ranked_metrics']['graded_ndcg_at_3'],
        'total_seconds': summary['runtime_seconds']['total'],
    }

sample_results = pd.DataFrame([sample_row(text_summary), sample_row(metadata_summary)])
display(sample_results)

,profile,scope,eval_cases,reference_cases,device,primary_accuracy,multilabel_micro_f1,multilabel_exact_match,ranked_primary_hit_at_3,graded_ndcg_at_3,total_seconds
0,embedding_knn_text_only_bge_small_en_v15,dev_sample,32,128,cpu,0.7188,0.7545,0.1875,0.9062,0.8405,15.078
1,embedding_knn_metadata_augmented_bge_small_en_v15,dev_sample,32,128,cpu,0.7812,0.7650,0.2500,0.9688,0.8812,15.269


## Explicit Run Commands

These are intentionally not run by default.

In [5]:
commands = [
    {
        'run_type': 'BGE sample text',
        'command': r'.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key bge_small_en_v15 --summary',
    },
    {
        'run_type': 'BGE sample metadata',
        'command': r'.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key bge_small_en_v15 --include-tags --summary',
    },
    {
        'run_type': 'BGE full dev text',
        'command': r'.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key bge_small_en_v15 --full-dev --summary',
    },
    {
        'run_type': 'BGE full dev metadata',
        'command': r'.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key bge_small_en_v15 --include-tags --full-dev --summary',
    },
    {
        'run_type': 'Qwen3 sample metadata GPU guarded',
        'command': r'.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key qwen3_embedding_06b --include-tags --device auto --require-cuda --batch-size 4 --summary',
    },
    {
        'run_type': 'Qwen3 full dev metadata GPU guarded',
        'command': r'.\.it_support\Scripts\python.exe scripts\evaluate_embedding_classifier_baseline.py --model-key qwen3_embedding_06b --include-tags --full-dev --device auto --require-cuda --batch-size 4 --summary',
    },
]
display(pd.DataFrame(commands))

,run_type,command
0,BGE sample text,.\.it_support\Scripts\python.exe scripts\evalu...
1,BGE sample metadata,.\.it_support\Scripts\python.exe scripts\evalu...
2,BGE full dev text,.\.it_support\Scripts\python.exe scripts\evalu...
3,BGE full dev metadata,.\.it_support\Scripts\python.exe scripts\evalu...
4,Qwen3 sample metadata GPU guarded,.\.it_support\Scripts\python.exe scripts\evalu...
5,Qwen3 full dev metadata GPU guarded,.\.it_support\Scripts\python.exe scripts\evalu...


## Working Decision

The downloaded-model workbench is ready. BGE kNN smoke samples are enough to validate the path and show that embeddings can improve secondary-label recall. Full-dev BGE, Qwen3 embedding classification, local LLM JSON classification, and fine-tuning should remain explicit heavier runs.